# Plotting rollout results in the Camera Dropbox environment
This notebook can be used to analyze the results of the experiments from `src/main.py` or `ipynb/ppo.ipynb`.

## Install dependencies
This notebook has been tested in Google Colab and the installation should work there. In a Jupyter notebook, you may have to set up a virtual environment and install the dependencies there instead.

In [ ]:
!apt-get install -y protobuf-compiler

In [ ]:
# Create the rollout_pb2.py file from rollout.proto.
!cd $MONA_PATH && protoc --python_out=. proto/rollout.proto

# Check that the new file was created succesfully.
!ls $MONA_PATH/proto/rollout_pb2.py

In [ ]:
import os

# You may have to adjust this path to make it point to the mona/ directory.
MONA_PATH = os.path.join(os.getcwd(), 'mona')
!ls $MONA_PATH

In [ ]:
# Create the rollout_pb2.py file from rollout.proto.
!cd $MONA_PATH && protoc --python_out=. proto/rollout.proto

# Check that the new file was created succesfully.
!ls $MONA_PATH/proto/rollout_pb2.py

In [ ]:
import sys
MONA_PATH = os.path.join(os.getcwd(), "mona")
# This will help Python find the mona.src files.
sys.path.insert(0, MONA_PATH)

In [ ]:
import copy
import dataclasses
import itertools
import os
import random
from typing import Any, Mapping, Sequence, Tuple
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from mona.src import data
from mona.src import file_handler as file_handler_lib
from mona.src import matrix_constructor
from mona.src import state as state_lib
from mona.proto import rollout_pb2

# Loading experiments

Each dictionary in the list `ALL_EXPERIMENT_PARAMS` represents a collection of related experiments. The experiments may have anywhere from 0 to 2 variables that change from experiment to experiment. The experiments should have already been run using the `value_iteration_main` Python script and had their data written to a directory; this Colab notebook simply loads existing data from that directory.

In later cells, the experiments will be graphed depending on the number of variables:
- 0 variables: A single graph
- 1 variable: A line of graphs
- 2 variables: A grid of graphs

If a dictionary specifies a field using a list of values, that field will be counted as a variable. Otherwise, the single value will be constant across all graphs.

If you don't specify a field, the default value for that field will be used. You can see the default value of each field in the "Setup" section.

## Setup

In [ ]:
file_handler = file_handler_lib.FileHandler()

# Whether to clear individual rollouts when loading experiments to save memory.
CLEAR_ROLLOUTS = True

@dataclasses.dataclass(frozen=True)
class RolloutExperiment:
  params: dict[str, Any]
  trials: list[rollout_pb2.RolloutIterations] | None = None
  rollout_array: np.ndarray | None = None


def get_rollout_experiment(params) -> RolloutExperiment:
  full_dir = file_handler.get_full_directory(
      data_dir=params.get(
          "data_dir",
          os.path.join(MONA_PATH, "data"),
      ),
      reward_function=params.get("reward_function", "bad"),
      per_step_penalty=params.get("per_step_penalty", 0.01),
      board_shape=params.get("board_shape", (4, 4)),
      max_boxes=params.get("max_boxes", 2),
      min_blocking_boxes=params.get("min_blocking_boxes", 1),
      init_vf_name=params.get("init_vf_name", "good"),
      init_noise_scale=params.get("init_noise", 0.0),
      explore_prob=params.get("explore_prob", None),
  )
  print("Loading from", full_dir)

  try:
    trials = file_handler.load_rollouts(full_dir)
    if CLEAR_ROLLOUTS:
      for trial in trials:
        for iteration in trial.iterations:
          del iteration.rollouts[:]
  except FileNotFoundError as e:
    raise FileNotFoundError(
        f"Rollout experiment with params {params} not found at {full_dir}"
    ) from e

  return RolloutExperiment(trials=trials, params=params)


def get_experiments_with_trials_dim(
    experiments: np.ndarray[RolloutExperiment],
) -> np.ndarray[RolloutExperiment]:
  # Assume that all experiments have the same number of trials.
  trials_length = len(experiments[0].trials)
  result = np.empty(experiments.size * trials_length, dtype=object)
  for i_e, experiment in enumerate(experiments):
    result[i_e * trials_length : (i_e + 1) * trials_length] = [
        RolloutExperiment(
            trials=[trial], params=experiment.params | {"trial": i_t + 1}
        )
        for i_t, trial in enumerate(experiment.trials)
    ]
  return result


def validate_num_trials(experiments: np.ndarray[RolloutExperiment]) -> None:
  trial_counts = []
  for experiment in experiments:
    trial_counts.append(len(experiment.trials))
  if not all([tc == trial_counts[0] for tc in trial_counts]):
    print(
        "\nWARNING: Expected all experiments to have the same number of"
        f" trials, but got {trial_counts}\n"
    )


def pad_iterations_in_experiments(
    experiments: np.ndarray[RolloutExperiment], use_global_max: bool
) -> None:
  """Add duplicate iterations to the end of trials to make them the same length."""
  pad_lengths = []
  for experiment in experiments:
    pad_lengths.append(
        max([len(trial.iterations) for trial in experiment.trials])
    )
  if use_global_max:
    pad_lengths = [max(pad_lengths)] * len(pad_lengths)

  for experiment, pad_length in zip(experiments, pad_lengths):
    for trial in experiment.trials:
      num_extra_iterations = pad_length - len(trial.iterations)
      extra_iterations = [trial.iterations[-1]] * num_extra_iterations
      trial.iterations.extend(extra_iterations)


def get_2d_array(arr: np.ndarray) -> np.ndarray:
  """Ensures a NumPy array has exactly 2 dimensions.

  Args:
      arr: The input NumPy array.

  Returns:
      The modified array with 2 dimensions if it had 0 or 1.

  Raises:
      ValueError: If the input array has 3 or more dimensions.
  """
  if arr.ndim == 0:  # Scalar
    return arr.reshape(1, 1)
  elif arr.ndim == 1:  # Vector
    return arr.reshape(-1, 1)
  elif arr.ndim == 2:  # Already 2D
    return arr
  else:
    raise ValueError("Input array must have 0, 1, or 2 dimensions.")

def load_rollouts_from_filepaths(params: dict[str, Any]) -> None:
  if "rollout_filepath" not in params:
    raise ValueError("rollout_filepath must be specified.")
  filepaths = params["rollout_filepath"]
  experiments = []
  for filepath, title in zip(filepaths, params["title"]):
    with open(filepath, "rb") as f:
      rollout_array = np.load(f)
    experiments.append(
        RolloutExperiment(
            params={**params, "rollout_filepath": filepath, "title": title},
            rollout_array=rollout_array
        )
    )
  experiments = np.array(experiments).reshape(-1, 1)
  return experiments, ["rollout_filepath"], None

def get_rollout_experiments(
    params: dict[str, Any],
    normalize_x_axis: bool,
    use_trial_as_dim: bool,
) -> tuple[np.ndarray[RolloutExperiment], list[str], matrix_constructor.MatrixConstructor]:
  if "rollout_filepath" in params:
    return load_rollouts_from_filepaths(params)

  # Get all variable parameters, in order of the numpy dims.
  var_params = {}

  for name, lst in params.items():
    if isinstance(lst, list) and len(lst) > 1:
      var_params[name] = lst
    else:
      # Use the single value directly in the fixed parameters.
      params[name] = lst[0] if isinstance(lst, list) else lst

  var_names = list(var_params.keys())

  experiments = np.array([
      get_rollout_experiment(
          {**params, **{name: value for name, value in zip(var_names, values)}}
      )
      for values in itertools.product(*var_params.values())
  ])

  validate_num_trials(experiments)

  pad_iterations_in_experiments(experiments, use_global_max=normalize_x_axis)

  if use_trial_as_dim:
    var_params["trial"] = list(range(1, len(experiments[0].trials) + 1))
    var_names.append("trial")
    experiments = get_experiments_with_trials_dim(experiments)

  experiments = experiments.reshape(
      [len(values) for values in var_params.values()]
  )

  # Ensure that the array is 2D so the experiments can be rendered in a grid.
  experiments = get_2d_array(experiments)

  mat_constructor = None
  if (
      "board_shape" not in params or not isinstance(params["board_shape"], list)
  ) and (
      "max_boxes" not in params or not isinstance(params["max_boxes"], list)
  ):
    mat_constructor = matrix_constructor.MatrixConstructor(
        board_shape=params.get("board_shape", (4, 4)),
        max_boxes=params.get("max_boxes", 2),
    )

  return experiments, var_names, mat_constructor

### Code for plot #1

In [ ]:
# 0 boxes = blue; 1 box = green; 2 boxes = orange.
milestone_colors = ["#a3b3d5", "#84ceb7", "#fca381"]
# 0 boxes = horizontal; 1 box = vertical; 2 boxes = diagonal.
milestone_hatch = ["--", "||", "//"]

# Make various changes to look more like the other plots, which use plotly.

# This style removes tick marks.
plt.style.use("seaborn-v0_8-white")

LINEWIDTH = 1.0
plt.rcParams["hatch.linewidth"] = LINEWIDTH

DARK_COLOR = "#262626"
GRAY_COLOR = "#808080"
plt.rcParams["text.color"] = DARK_COLOR
plt.rcParams["xtick.color"] = DARK_COLOR
plt.rcParams["ytick.color"] = DARK_COLOR
plt.rcParams["axes.labelcolor"] = GRAY_COLOR
plt.rcParams["hatch.color"] = GRAY_COLOR

# Make the x and y labels the same size as the rest of the text.
plt.rcParams["figure.labelsize"] = "medium"


def get_experiment_info(experiment: RolloutExperiment) -> str:
  if "title" in experiment.params:
    return experiment.params["title"]

  info_strs = []
  for name in var_names:
    value = experiment.params[name]
    if name == "init_noise":
      name = "ε"
    if name == "explore_prob":
      name = "p_ₑₓₚₗₒᵣₑ"
    info_str = f"{name}={value}"
    if name == "min_blocking_boxes":
      info_str = f"{value} boxes needed to block" if value > 1 else f"{value} box needed to block"
    if name == "board_shape":
      info_str = f"{value[0]}×{value[1]} grid"
    if name == "init_policy_name":
      info_str = {
          "acc_0.111": "initial policy R = 0.111",
          "acc_0.366": "initial policy R = 0.366",
          "acc_0.821": "initial policy R = 0.821",
      }.get(value, value)
    info_strs.append(info_str)
  return ", ".join(info_strs)


def get_data_shape(experiment: RolloutExperiment) -> tuple[int, int, int]:
  num_trials = len(experiment.trials)
  num_box_counts = 0
  num_iterations = 0
  for trial in experiment.trials:
    num_iterations = max(num_iterations, len(trial.iterations))
    for iteration in trial.iterations:
      num_box_counts = max(num_box_counts, len(iteration.stats.box_milestones))

  return num_trials, num_iterations, num_box_counts

def maybe_add_title(experiment: RolloutExperiment, ax, title_pos: str | None):
  if title_pos is not None:
    title = get_experiment_info(experiment)
    if title_pos == "top":
      ax.text(
          0.5,
          1.05,
          title,
          horizontalalignment="center",
          verticalalignment="bottom",
          transform=ax.transAxes,
      )
    elif title_pos == "right":
      ax.text(
          1.05,
          0.5,
          title,
          horizontalalignment="left",
          verticalalignment="center",
          transform=ax.transAxes,
      )
    else:
      raise ValueError(f"Invalid title_pos: {title_pos}")

def handle_ticks(ax, x_values, add_xticks: bool, add_yticks: bool):
  if not add_xticks:
    ax.set_xticks([])
  # Quick fix: this ensures that the xticks will end on exactly 50.
  # Otherwise, it would use [0, 20, 40].
  elif 50.0 not in ax.get_xticks() and np.max(x_values) == 50:
    ax.set_xticks([0, 25, 50])

  if not add_yticks:
    ax.set_yticks([])

def get_percentage_data(experiment: RolloutExperiment, average_results=True):
  if experiment.rollout_array is not None:
    rollout_array = experiment.rollout_array
    if rollout_array.ndim == 3:
      if average_results:
        rollout_array = rollout_array.mean(axis=0)
      else:
        best_i = np.argmax(rollout_array[0:3, -1, 2], axis=0)
        rollout_array = rollout_array[best_i]
    assert rollout_array.ndim == 2
    percentage_data = rollout_array / np.sum(rollout_array, axis=-1, keepdims=True)
    x_values = np.arange(rollout_array.shape[0]) * experiment_params.get(
      "save_every_nth_iteration", 1
    )
    return x_values, percentage_data

  num_trials, num_iterations, num_box_counts = get_data_shape(experiment)
  # Start with all np.nan to account for iterations that don't exist
  # (we don't run PPO for every single value of M).
  percentage_data = np.full(
      (num_trials, num_iterations, num_box_counts), np.nan
  )
  for t, trial in enumerate(experiment.trials):
    for i, iteration in enumerate(trial.iterations):
      if iteration.stats.num_initial_states == 0:
        continue
      for b in range(num_box_counts):
        # We have to explicitly handle box milestones that aren't reached so
        # that they'll be counted as 0 instead of np.nan.
        if b < len(iteration.stats.box_milestones):
          percentage_data[t, i, b] = (
              iteration.stats.box_milestones[b].num_initial_states
              / iteration.stats.num_initial_states
          )
        else:
          percentage_data[t, i, b] = 0
  percentage_data = np.nanmean(percentage_data, axis=0)

  data_exists = ~np.all(np.isnan(percentage_data), axis=1)
  data_exists_idxs = np.where(data_exists)[0]
  # Get only the existing data and convert nan to 0.
  percentage_data = np.nan_to_num(percentage_data[data_exists])

  # Multiply each index by save_every_nth_iteration to get the values to plot.
  x_values = data_exists_idxs * experiment_params.get(
      "save_every_nth_iteration", 1
  )

  return x_values, percentage_data


def condense_unsafe(percentage_data):
  # Count all milestones for 2 or more boxes together as "unsafe behavior"
  unsafe_sum = np.sum(percentage_data[:, 2:], axis=1, keepdims=True)
  return np.column_stack((percentage_data[:, :2], unsafe_sum))


def plot_grid(
    experiments,
    plot_fn,
    xlabel=None,
    ylabel=None,
    legend=True,
    legend_spacing=0.4,
    aspect_ratio=1.618,  # golden ratio
    padding=1.08,
    **kwargs,
):
  cell_height = 3.0
  figsize = (
      cell_height * aspect_ratio * experiments.shape[0],  # width
      cell_height * experiments.shape[1],  # height
  )

  fig, axes = plt.subplots(
      *experiments.T.shape,
      figsize=figsize,
      squeeze=False,
  )

  used_labels = set()  # This prevents duplicate entries in the legend.
  for row in range(experiments.shape[0]):
    for col in range(experiments.shape[1]):
      plot_fn(experiments[row, col], axes[col, row], used_labels, **kwargs)

  fig.supxlabel(xlabel)
  fig.supylabel(ylabel)
  plt.tight_layout(pad=padding)

  # fig.subplots_adjust(top=fig.subplotpars.top - title_height / figsize[1])
  if legend:
    fig.legend(
        loc="lower center",
        frameon=False,
        ncol=len(used_labels),
        bbox_to_anchor=(0.5, -legend_spacing / figsize[1]),
    )
  plt.show()

  return fig, axes


def plot_big_with_little_row(
    experiments,
    plot_fn,
    xlabel=None,
    ylabel=None,
    legend=True,
    legend_spacing=0.4,
    aspect_ratio=1.618,  # golden ratio
    **kwargs,
):
  assert (
      experiments.shape[1] == 1
  ), "This function expects a single column of experiments."
  num_graphs = experiments.shape[0]
  num_small_graphs = num_graphs - 1
  cell_height = 3.0
  big_cell_width = cell_height * aspect_ratio
  # Adjust dimensions for horizontal layout
  fig_width = big_cell_width
  fig_height = cell_height * (1 + 1 / num_small_graphs)
  fig = plt.figure(figsize=(fig_width, fig_height), constrained_layout=True)
  grid_shape = (num_small_graphs + 1, num_small_graphs)
  # Create the big plot
  ax_main = plt.subplot2grid(
      grid_shape,
      (0, 0),
      rowspan=num_small_graphs,
      colspan=num_small_graphs,
  )
  used_labels = set()
  plot_fn(experiments[0, 0], ax_main, used_labels, title_pos="top", **kwargs)
  # Create the small plots in the bottom row
  small_axes = []
  for i in range(1, num_graphs):
    ax_small = plt.subplot2grid(grid_shape, (grid_shape[0] - 1, i - 1))
    # Remove the yticks from all but the leftmost graph, to save space
    add_yticks = i == 1
    plot_fn(
        experiments[i, 0],
        ax_small,
        used_labels,
        title_pos="top",
        add_xticks=True,
        add_yticks=add_yticks,
        **kwargs,
    )
    small_axes.append(ax_small)

  fig.supxlabel(xlabel)
  fig.supylabel(ylabel)

  if legend:
    legend_offset = -legend_spacing / fig_height
    fig.legend(
        loc="lower center",
        frameon=False,
        bbox_to_anchor=(0.5, legend_offset),
    )
  plt.show()
  return fig, [ax_main] + small_axes

def get_labels_if_unused(candidate_labels, used_labels):
  return_str = False
  if isinstance(candidate_labels, str):
    return_str = True
    candidate_labels = [candidate_labels]
  result = []
  for candidate_label in candidate_labels:
    if candidate_label in used_labels:
      result.append(None)
    else:
      used_labels.add(candidate_label)
      result.append(candidate_label)
  return result[0] if return_str else result


def plot_box_milestone_percentages(
    experiment: RolloutExperiment | np.ndarray,
    ax: plt.Axes,
    used_labels: set[str],
    title_pos: str = "top",
    add_xticks=True,
    add_yticks=True,
):
  """Plots the percentage of initial states reaching each box milestone over time."""
  data_exists_idxs, percentage_data = get_percentage_data(experiment)
  # Count any milestone with ≥2 boxes as all "Unsafe behavior"
  condensed_data = condense_unsafe(percentage_data)

  # If there are more than 2 boxes that could be pushed in, use ≥ in the label.
  unsafe_label = (
      "Unsafe behavior (≥2 boxes)"
      if "min_blocking_boxes" in var_names
      else "Unsafe behavior (2 boxes)"
  )
  # Show the main groups of results: failure, desired, and unsafe behavior.
  ax.stackplot(
      data_exists_idxs,
      condensed_data.T,
      labels=get_labels_if_unused(
          [
              "Failure (0 boxes)",
              "Desired behavior (1 box)",
              unsafe_label,
          ],
          used_labels,
      ),
      colors=milestone_colors,
      hatch=milestone_hatch,
      edgecolor=GRAY_COLOR,
      linewidth=LINEWIDTH,
  )
  # Make dotted lines to show any box milestones past 2.
  for i in range(3, percentage_data.shape[1]):
    cum_percentage = np.sum(percentage_data[:, :i], axis=1)
    # Get the first index.
    first_idx_to_plot = (cum_percentage < 0.99).argmax() - 1
    ax.plot(
        data_exists_idxs[first_idx_to_plot:],
        cum_percentage[first_idx_to_plot:],
        color="black",
        linestyle=":",
        label=get_labels_if_unused("Additional boxes", used_labels),
    )

  handle_ticks(ax, data_exists_idxs, add_xticks, add_yticks)
  # ax.set_xticks([0, 50000, 100000, 150000, 200000])
  maybe_add_title(experiment, ax, title_pos)

  # The plot should extend exactly to the maximum value on each axis.
  # By default, it would add some extra padding space.
  ax.set_xlim(left=0, right=np.max(data_exists_idxs))
  ax.set_ylim(bottom=0, top=np.max(condensed_data))
  for spine in ax.spines.values():
    spine.set_visible(False)

### Code for plot #2

In [ ]:
def plot_two_graphs_with_all_experiments(
    experiments,
    plot_fn,
    xlabel=None,
    ylabel=None,
    legend=True,
    legend_spacing=0.4,
    aspect_ratio=1.618,  # golden ratio
    padding=1.08,
    **kwargs,
):
  assert experiments.shape[1] == 1
  num_graphs = 2
  cell_height = 3.0
  figsize = (
      cell_height * aspect_ratio * num_graphs,  # width
      cell_height,  # height
  )

  fig, axes = plt.subplots(
      1, 2,
      figsize=figsize,
      squeeze=False,
  )

  used_labels = set()  # This prevents duplicate entries in the legend.
  colors = ['#ae451f', '#999999']  # Orange and grey
  labels = ['MONA', 'Ordinary RL']
  for i in range(num_graphs):
    plot_fn(
        experiments[i, 0],
        left_ax=axes[0, 0],
        right_ax=axes[0, 1],
        used_labels=used_labels,
        color=colors[i],
        label=labels[i],
        **kwargs,
    )

  axes[0, 0].set_title("Observed return")
  axes[0, 1].set_title("Ground truth return")

  fig.supxlabel(xlabel)
  fig.supylabel(ylabel)
  plt.tight_layout(pad=padding)

  if legend:
    fig.legend(
        loc="lower center",
        frameon=False,
        ncol=len(used_labels),
        bbox_to_anchor=(0.5, -legend_spacing / figsize[1]),
    )
  plt.show()

  return fig, axes

def plot_rollout_rewards(
    experiment: RolloutExperiment | np.ndarray,
    left_ax: plt.Axes,
    right_ax: plt.Axes,
    used_labels: set[str],
    color: str,
    label: str,
    add_xticks=True,
    add_yticks=True,
):
  """Plots the percentage of initial states reaching each box milestone over time."""
  data_exists_idxs, percentage_data = get_percentage_data(experiment)
  condensed_data = condense_unsafe(percentage_data)

  individual_percentage_data = []
  for rollout_array in experiment.rollout_array:
    rollout_array = np.expand_dims(rollout_array, 0)
    _, ind_pct_data = get_percentage_data(
        RolloutExperiment(params={}, rollout_array=rollout_array)
    )
    individual_percentage_data.append(ind_pct_data)
  individual_percentage_data = np.array(individual_percentage_data)

  def get_observed_return(arr):
    return np.sum(arr.T[1:], axis=0)

  def get_ground_truth_return(arr):
    return arr.T[1]

  for ax, data_fn in zip(
      [left_ax, right_ax],
      [get_observed_return, get_ground_truth_return],
  ):
    ax.plot(
        data_exists_idxs,
        data_fn(percentage_data),
        label=get_labels_if_unused([label], used_labels)[0],
        color=color,
    )
    for ip_data in individual_percentage_data:
      ax.plot(
          data_exists_idxs,
          data_fn(ip_data),
          color=color,
          alpha=0.2,
      )

# Choose the experiment to plot

You can add your own experiments with any parameters to the `ALL_EXPERIMENT_PARAMS` dictionary below. Using this notebook and data from `src/main.py` and `ipynb/ppo.ipynb`, you should be able to replicate any graph from the paper.

In [ ]:
ALL_EXPERIMENT_PARAMS = {
    # These experiments can be replicated with this command, with --noise_scale either 0 or 0.5:
    #
    # bazel build //src:main && \
    # bazel-bin/src/main --board_shape=4,4 --max_boxes=2 --episode_step_limit=50 \
    #   --reward_function=bad --per_step_penalty=0.01 --initial_vf=good \
    #   --noise_scale=(0 | 0.5) --data_dir=data/ --save_rollouts_level=3
    #
    # You can use --num_trials to get several trials, which will be averaged.
    "VI: Optimal safe VF with noise": {
        "init_noise": [0.0, 0.5],
    },
    # These files came from the PPO notebook. Their names were shortened for convenience.
    "PPO: MONA vs. Ordinary RL": {
        "rollout_filepath": [
            os.path.join(MONA_PATH, "data/ppo_output/mona_ppo_rollouts.npy"),
            os.path.join(MONA_PATH, "data/ppo_output/ordinary_rl_ppo_rollouts.npy"),
        ],
        "title": [
            "MONA",
            "Ordinary RL",
        ],
        "save_every_nth_iteration": 10000
    },
}

# Uncomment whichever experiment you want to make plots for.
experiment_params = ALL_EXPERIMENT_PARAMS[
    "PPO: MONA vs. Ordinary RL"
    # "VI: Optimal safe VF with noise"
]

# If True, each trial will be plotted in its own graph.
# Otherwise, the results from all trials will be averaged together.
USE_TRIAL_AS_DIM = False

# If True, all graphs will be "padded" with their last state until the maximum
# x-axis value range across all experiments, to make them easier to compare
# visually.
# Otherwise, graphs may have different maximum x values.
NORMALIZE_X_AXIS = True

experiments, var_names, mat_constructor = get_rollout_experiments(
    experiment_params,
    normalize_x_axis=NORMALIZE_X_AXIS,
    use_trial_as_dim=USE_TRIAL_AS_DIM,
)

print("Experiments shape:", experiments.shape)
print("Variable names:", var_names)

# Plot the behavior distribution over time

In [ ]:
PLOT_GRID = True

xlabel = "Training steps" if "rollout_filepath" in experiment_params else "Optimization horizon"

if PLOT_GRID:
  fig, axes = plot_grid(
      experiments,
      plot_box_milestone_percentages,
      xlabel=xlabel,
      ylabel="Behavior distribution",
      legend_spacing=0.4,
      aspect_ratio=1.618,
      # padding=1.8  # We may need to increase the padding for some plots
  )
else:
  fig, axes = plot_big_with_little_row(
      experiments[:, :1],  # Ignore any experiment rows after the first.
      plot_box_milestone_percentages,
      xlabel=xlabel,
      ylabel="Behavior distribution",
      legend_spacing=0.9,
      aspect_ratio=1.618,
  )

# Uncomment this to save your figure.
# fig.savefig("<your_file>.pdf", bbox_inches="tight")

# Plot MONA vs. Ordinary RL
This will only work for plots where there are two experiments side-by-side, like "MONA PPO, M=1" above.

In [ ]:
xlabel = "Training steps" if "rollout_filepath" in experiment_params else "Optimization horizon"

fig, axes = plot_two_graphs_with_all_experiments(
    experiments,
    plot_rollout_rewards,
    xlabel=xlabel,
    ylabel="Return",
    legend_spacing=0.4,
    aspect_ratio=1.618,
    # padding=1.8  # We may need to increase the padding for some plots
)

# Uncomment this to save your figure.
# fig.savefig("<your_file>.pdf", bbox_inches="tight")